# GAN Variants and Modern Practice

If the original GAN chapter introduces the adversarial game, this chapter should explain why one almost never stops at the vanilla minimax loss in practice. The standard GAN is elegant, but its optimization signal can be fragile, the discriminator can become too sharp or too unstable, and the entire game can fail to cover the true data distribution well.

For teaching purposes, it is better to study **a small number of central GAN variants well** than to survey a large zoo. We therefore focus on three common branches: **WGAN-GP**, **spectral normalization**, and **conditional GANs**. Together, they cover the three main axes of adversarial modeling: better geometry, better critic control, and controlled generation.

## From the Vanilla GAN to Wasserstein Geometry

The most important conceptual shift in the GAN literature is the move from the Jensen-Shannon geometry of the original game to the **Wasserstein-1 distance**. In dual form,
```{math}
W_1(p_{gt}, p_\theta)
=
\sup_{\|f\|_L \le 1}
\mathbb{E}_{\boldsymbol{x}\sim p_{gt}}[f(\boldsymbol{x})]
-
\mathbb{E}_{\boldsymbol{x}\sim p_\theta}[f(\boldsymbol{x})].
```
The discriminator is replaced by a **critic**, and the key mathematical condition becomes **1-Lipschitz continuity**.

Why is this such a big deal? Because Wasserstein distance can still change smoothly even when the supports of the real and generated distributions barely overlap. That is precisely the regime where the original GAN often struggles most, especially early in training. In that sense, WGAN is not just a variant with a different formula. It is a different view of what signal the generator ought to receive.

A simple mental picture helps. Imagine that the true data are images of shoes and the generator initially produces something like amorphous grayscale blobs. Under the original adversarial objective, the discriminator may confidently separate the two distributions, leaving little useful gradient structure. Under the Wasserstein view, the critic is instead encouraged to estimate **how far probability mass must move** to turn the blobs into plausible shoes. That tends to produce a more graded training signal.

The catch is that the theory requires the critic to be 1-Lipschitz. The original WGAN enforced this with **weight clipping**, which is historically important but often a poor numerical approximation. That leads directly to the most standard practical version.

## WGAN-GP

The best-known practical refinement is **WGAN-GP**, which adds a **gradient penalty** on interpolated samples:
```{math}
\lambda\,\mathbb{E}_{\hat{\boldsymbol{x}}}
\left[(\|\nabla_{\hat{\boldsymbol{x}}} D(\hat{\boldsymbol{x}})\|_2 - 1)^2\right].
```
This asks the critic to change at approximately unit speed along directions between real and fake points. Pedagogically, this is one of the most useful GAN formulas to unpack carefully. It turns an abstract Lipschitz constraint into a concrete regularizer that students can both understand and implement.

WGAN-GP also changes the training loop in an instructive way. The critic must backpropagate through its **inputs** to compute the gradient penalty, which makes the implementation materially different from the vanilla GAN. This is exactly why it deserves a dedicated code section rather than a passing note.

## Spectral Normalization

**Spectral normalization** is a different answer to the same broad question: how do we keep the critic from becoming too wild? Instead of adding a penalty term, we constrain each layer by dividing its weight matrix by an estimate of its largest singular value.

This makes spectral normalization one of the cleanest examples of an **architectural regularization** technique. The loss can stay almost the same, yet the critic class becomes much better behaved. In practice, this is attractive because it is easy to add and often stabilizes training with very little extra machinery.

From a teaching perspective, spectral normalization is valuable because it broadens the students' mental model. GAN stability is not only about choosing the right objective. It is also about controlling the geometry of the discriminator network itself.

## Conditional GANs

A third standard branch is the **conditional GAN (cGAN)**. Here the target is no longer an unconditional image distribution, but a conditional one. The generator receives noise together with some condition $\boldsymbol{y}$, and the discriminator judges both realism and condition compatibility:
```{math}
\min_G \max_D
\mathbb{E}_{(\boldsymbol{x},\boldsymbol{y})\sim p_{gt}}[\log D(\boldsymbol{x},\boldsymbol{y})]
+
\mathbb{E}_{\boldsymbol{z}\sim p(\boldsymbol{z}),\boldsymbol{y}\sim p(\boldsymbol{y})}
[\log(1-D(G(\boldsymbol{z},\boldsymbol{y}),\boldsymbol{y}))].
```

This is important because it shows that adversarial learning is not just a tool for sampling from noise. It is a tool for **controlled synthesis**. If $\boldsymbol{y}$ is a class label, the model should produce an image of the requested class. If $\boldsymbol{y}$ is another image, one enters the larger family of image-to-image translation models.

A simple example makes the distinction vivid. Suppose $\boldsymbol{y}$ says “dress.” The latent noise should still determine style, texture, or silhouette, but the output should not drift into “ankle boot” or “bag.” In other words, conditional GANs split the synthesis problem into two parts: the condition specifies **what kind of object** must be generated, while the noise specifies **which instance** of that kind appears.

This is one of the best points in the course to connect GANs to the later diffusion chapter on prompted generation. The conditioning mechanism is different in detail, but the high-level question is the same: how do we combine **control** with **stochastic variability**?

## A Brief Outlook Beyond the Core Set

There are, of course, many other GAN variants: **CycleGAN** for unpaired translation, **StyleGAN** for highly structured latent control, and large class-conditional systems such as **BigGAN**. These matter historically and practically, but for a first three-hour GAN block it is usually better to treat them as extensions of the three central ideas already discussed.

- CycleGAN extends the conditional viewpoint to unpaired domain translation.
- StyleGAN extends the architectural-control viewpoint to multi-scale latent modulation.
- BigGAN extends the conditional and stabilization story to large-scale class-conditioned generation.

Once students understand WGAN-GP, spectral normalization, and cGANs, they are in a good position to read these larger models without getting lost.

## What To Remember

A manageable but meaningful GAN module can be organized around three questions.

1. How do we improve the **geometry of the training signal**? Answer: **WGAN-GP**.
2. How do we control the **critic itself**? Answer: **spectral normalization**.
3. How do we move from unconditional sampling to **controlled generation**? Answer: **conditional GANs**.

That is a better backbone for teaching than a long survey of names. These variants are common, conceptually distinct, and concrete enough that they can and should appear in the implementation notebook as well.